A — Imports

In [21]:
import random
import json
from pathlib import Path

import torch
import matplotlib.pyplot as plt
import torchvision.transforms as T
from torch.utils.data import DataLoader

from modules.tokenizers import Tokenizer
from modules.datasets import IuxrayMultiImageDataset
from models.models import BaseCMNModel

B — Re-create args

In [22]:
args.dataset_name = "iu_xray"
args.threshold = getattr(args, "threshold", 3)
args.max_seq_length = getattr(args, "max_seq_length", 60)

args.visual_extractor = getattr(args, "visual_extractor", "resnet101")
args.visual_extractor_pretrained = getattr(args, "visual_extractor_pretrained", True)

# Common CMN defaults
args.num_layers = getattr(args, "num_layers", 3)
args.num_heads  = getattr(args, "num_heads", 8)
args.d_model    = getattr(args, "d_model", 512)
args.d_ff       = getattr(args, "d_ff", 512)
args.d_vf       = getattr(args, "d_vf", 2048)
args.dropout    = getattr(args, "dropout", 0.1)
args.drop_prob_lm = getattr(args, "drop_prob_lm", 0.1)
args.use_bn     = getattr(args, "use_bn", 0)

# CMN memory
args.topk     = getattr(args, "topk", 32)
args.cmm_size = getattr(args, "cmm_size", 2048)   # <-- this is the one that caused your mismatch earlier
args.cmm_dim  = getattr(args, "cmm_dim", 512)

# Tokenizer
tokenizer = Tokenizer(args)

tokenizer.pad_idx = 0
tokenizer.bos_idx = 0
tokenizer.eos_idx = 0
tokenizer.unk_idx = tokenizer.token2idx.get("<unk>", 0)

args.pad_idx = 0
args.bos_idx = 0
args.eos_idx = 0
args.unk_idx = tokenizer.unk_idx

print("Tokenizer ready | vocab:", tokenizer.get_vocab_size())

✅ Tokenizer ready | vocab: 928


C — Image transforms

In [23]:
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

test_dataset = IuxrayMultiImageDataset(args=args, tokenizer=tokenizer, split="test", transform=transform)

idx = random.randint(0, len(test_dataset) - 1)
uid, images, report_ids, report_masks, seq_len = test_dataset[idx]

print("Random UID:", uid, "| idx:", idx)
print("images:", images.shape, "| seq_len:", seq_len)

✅ Random UID: 1775 | idx: 30
images: torch.Size([2, 3, 224, 224]) | seq_len: 27


D — Tokenizer

In [24]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CKPT_PATH = Path("/home/psivakumar/thesis/code/R2GenCMN/outputs/iu_xray_r2gencmn_baseline_5ep/current_checkpoint.pth")
assert CKPT_PATH.exists(), f"Checkpoint not found: {CKPT_PATH}"

model = BaseCMNModel(args, tokenizer).to(DEVICE)

ckpt = torch.load(str(CKPT_PATH), map_location=DEVICE)
state_dict = ckpt["state_dict"] if isinstance(ckpt, dict) and "state_dict" in ckpt else ckpt

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print("Model loaded")
print("Missing keys:", len(missing), "| Unexpected keys:", len(unexpected))

model.eval()

/home/psivakumar/.conda/envs/ml/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/psivakumar/.conda/envs/ml/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/home/psivakumar/.conda/envs/ml/lib/python3.10/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/tmp/ipykernel_553308/2555607884.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle mo

RuntimeError: Error(s) in loading state_dict for BaseCMNModel:
	size mismatch for encoder_decoder.model.encoder.layers.0.feed_forward.w_1.weight: copying a param with shape torch.Size([512, 512]) from checkpoint, the shape in current model is torch.Size([2048, 512]).
	size mismatch for encoder_decoder.model.encoder.layers.0.feed_forward.w_1.bias: copying a param with shape torch.Size([512]) from checkpoint, the shape in current model is torch.Size([2048]).
	size mismatch for encoder_decoder.model.encoder.layers.0.feed_forward.w_2.weight: copying a param with shape torch.Size([512, 512]) from checkpoint, the shape in current model is torch.Size([512, 2048]).
	size mismatch for encoder_decoder.model.encoder.layers.1.feed_forward.w_1.weight: copying a param with shape torch.Size([512, 512]) from checkpoint, the shape in current model is torch.Size([2048, 512]).
	size mismatch for encoder_decoder.model.encoder.layers.1.feed_forward.w_1.bias: copying a param with shape torch.Size([512]) fro

E — Test Dataset 

In [17]:
from modules.datasets import IuxrayMultiImageDataset

test_dataset = IuxrayMultiImageDataset(
    args=args,
    tokenizer=tokenizer,
    split="test",
    transform=transform
)

print("test_dataset size:", len(test_dataset))

✅ test_dataset size: 44


F — Sanity Check (CRITICAL)

In [19]:
from pathlib import Path
import json

ANN_PATH = args.ann_path
ann = json.loads(Path(ANN_PATH).read_text())

# take first test example and first image path
ex0 = ann["test"][0]
rel0 = ex0["image_path"][0] 

candidates = [
    "/home/psivakumar/thesis/data/iu_xray",    
    "/home/psivakumar/thesis/data/iu_xray/images", 
    "/home/psivakumar/thesis/data/iu_xray/images/images_normalized",
    "/home/psivakumar/thesis/code/R2GenCMN/data/iu_xray",
]

print("Testing relative path from JSON:", rel0)
found = None
for root in candidates:
    p = Path(root) / rel0
    ok = p.exists()
    print(f"{root} + {rel0} -> exists={ok}")
    if ok:
        found = root
        break

assert found is not None, "None of the candidate roots worked. We'll need to locate where 'images/' lives."

args.image_dir = found
print("\nUsing args.image_dir =", args.image_dir)
print("Example resolved path:", (Path(args.image_dir) / rel0))

Testing relative path from JSON: images/1623_IM-0405-4004.dcm.png
/home/psivakumar/thesis/data/iu_xray + images/1623_IM-0405-4004.dcm.png -> exists=False
/home/psivakumar/thesis/data/iu_xray/images + images/1623_IM-0405-4004.dcm.png -> exists=False
/home/psivakumar/thesis/data/iu_xray/images/images_normalized + images/1623_IM-0405-4004.dcm.png -> exists=False
/home/psivakumar/thesis/code/R2GenCMN/data/iu_xray + images/1623_IM-0405-4004.dcm.png -> exists=True

✅ Using args.image_dir = /home/psivakumar/thesis/code/R2GenCMN/data/iu_xray
Example resolved path: /home/psivakumar/thesis/code/R2GenCMN/data/iu_xray/images/1623_IM-0405-4004.dcm.png


In [20]:
import torchvision.transforms as T
from torch.utils.data import DataLoader
from modules.datasets import IuxrayMultiImageDataset

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

test_dataset = IuxrayMultiImageDataset(args=args, tokenizer=tokenizer, split="test", transform=transform)
test_loader  = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=0)

uid, images, report_ids, report_masks, seq_len = test_dataset[0]
print("Loaded one sample")
print("UID:", uid)
print("images:", images.shape)
print("seq_len:", seq_len)

✅ Loaded one sample
UID: 1623
images: torch.Size([2, 3, 224, 224])
seq_len: 21
